# 00 · Orientation

This tutorial teaches Python by translating from R. Every chapter follows the same pattern:

> **In R** — a snippet you already know, shown for reference (not run).
>
> **In Python** — the equivalent, in a cell you run yourself.

This first chapter has no real data analysis in it. It covers the things that are different *around* the code: the editor, how packages are loaded, how a project's packages are managed, and how to get help. It ends with a short preview of where the tutorial is headed.

To run a cell, click into it and press **Shift+Enter**.

## Check the setup

Run the cell below. The second line should be a path ending in `python-r-tutorial/.venv/...`. If it points somewhere else, the notebook is using the wrong Python: pick the `.venv` interpreter from the kernel selector at the top right of the notebook.

In [ ]:
import sys

print(sys.version)
print(sys.executable)

## Positron vs RStudio

Positron is built by Posit, the company behind RStudio, and most of the layout carries over.

| RStudio | Positron |
| --- | --- |
| Console | Console |
| Environment pane | Variables pane |
| `View(df)` | Data Explorer (click a data frame in the Variables pane) |
| Plots pane | Plots pane |
| Help pane | Help pane |
| `.R` script | `.py` script |
| `.Rmd` / `.qmd` notebook | `.ipynb` notebook (this file) |
| Ctrl/Cmd+Enter runs the current line | Same, in a `.py` script |
| Ctrl/Cmd+Shift+Enter runs the chunk | Shift+Enter runs the cell |

Two things are worth knowing about notebooks:

- A notebook keeps one Python session alive, like an R session. Variables defined in one cell are available in every cell you run afterwards.
- Cells run in the order *you* run them, not top to bottom. If results look strange, use **Restart** and **Run All** to get a clean slate, the same way you would restart R.

Outside of tutorials, a lot of everyday Python is written in plain `.py` scripts and run line by line into the console, exactly as you would with an `.R` script.

## Packages and imports

**In R**

```r
library(dplyr)

penguins <- read.csv("../data/penguins.csv")
head(penguins)
```

`library(dplyr)` dumps every dplyr function into your session, so you call `filter()` and `mutate()` directly.

**In Python**

Importing a package gives you one name, and everything inside it is reached with a dot. `pd.read_csv` means "the `read_csv` function that lives in pandas", the same idea as `readr::read_csv` in R, except that in Python the prefix is the norm rather than the exception.

In [ ]:
import pandas as pd

penguins = pd.read_csv("../data/penguins.csv")
penguins.head()

A few things to notice:

- `import pandas as pd` loads pandas and gives it the nickname `pd`. The nicknames are conventions that everyone follows: `pd` for pandas, `np` for NumPy, `plt` for matplotlib's plotting module, `sns` for seaborn.
- Assignment is `=`. There is no `<-`.
- The first row is labelled `0`, not `1`. Python counts from zero. Chapter 01 covers this properly.
- The last expression in a cell is displayed automatically, like typing a name at the R console.

There are three common forms of `import`:

In [ ]:
import numpy                     # use as numpy.mean(...)
import numpy as np               # use as np.mean(...)        <- the usual form
from numpy import mean, median   # use as mean(...)           <- closest to library()

np.mean([1, 2, 3, 4])

## Functions vs methods

This is the biggest change in how code *reads*.

**In R**

```r
head(penguins)
nrow(penguins)
dim(penguins)
names(penguins)
```

In R nearly everything is a function that you pass the object into.

**In Python**

Most operations are *methods*: functions attached to the object, called with a dot after it. Some facts about an object are plain *attributes*, with no parentheses.

In [ ]:
print(len(penguins))      # a function, like nrow()
print(penguins.shape)     # an attribute, like dim(): no parentheses
print(penguins.columns)   # an attribute, like names()

In [ ]:
penguins.tail(3)          # a method, like tail(penguins, 3)

The upside is discoverability. Type `penguins.` in a cell and press **Tab** to list everything a data frame can do. It serves the same purpose as browsing a package's function index in R.

A rough rule: if it takes parentheses, it *does* something (`.head()`, `.describe()`); if it does not, it *is* something (`.shape`, `.columns`).

## Getting help

**In R**

```r
?mean
help(mean)
```

**In Python**

Run the cell below, then look at the **Help** pane, not under the cell. Positron sends `help()` there, the same way RStudio handles `?mean`, so the cell itself shows no output. (In JupyterLab the text prints below the cell instead.)

In [ ]:
help(len)

The R-style shortcut also works in a notebook or the console: `len?` or `pd.read_csv?`.

The online documentation is where most people end up:

- pandas: <https://pandas.pydata.org/docs/>
- NumPy: <https://numpy.org/doc/stable/>
- plotnine: <https://plotnine.org/>
- seaborn: <https://seaborn.pydata.org/>

## Environments: renv vs uv

In R, packages are usually installed once into a shared library, and `renv` is an optional extra for pinning versions per project. In Python, one isolated environment *per project* is the standard practice, because different projects routinely need incompatible versions of the same package.

This project uses `uv` for that. You already ran `uv sync`, which created the `.venv/` folder.

| R | Python (uv) | |
| --- | --- | --- |
| `DESCRIPTION` | `pyproject.toml` | the packages the project asks for |
| `renv.lock` | `uv.lock` | the exact versions that were installed |
| `renv/library/` | `.venv/` | where the packages live |
| `renv::restore()` | `uv sync` | rebuild the environment from the lockfile |
| `install.packages("x")` + `renv::snapshot()` | `uv add x` | install a package and record it |
| `Rscript analysis.R` | `uv run python analysis.py` | run a script in the project environment |

These are terminal commands, not Python code. Run them in Positron's **Terminal** tab from the project folder.

You will also see `pip install x` and `conda install x` all over the internet. They are older tools for the same job. In this project, use `uv add x` wherever instructions say `pip install x`.

## A preview of what's ahead

Here is a small analysis in both languages. Don't worry about the details of the Python yet; chapters 03 and 05 take it apart piece by piece. The point is that the *shape* of the work is the same.

**In R**

```r
library(dplyr)

penguins |>
  filter(!is.na(body_mass_g)) |>
  group_by(species) |>
  summarise(
    n = n(),
    mean_mass = mean(body_mass_g)
  ) |>
  arrange(desc(mean_mass))
```

**In Python**

The pipe becomes a chain of methods. Wrapping the chain in parentheses lets it span several lines.

In [ ]:
(
    penguins
    .dropna(subset=["body_mass_g"])
    .groupby("species")
    .agg(
        n=("body_mass_g", "size"),
        mean_mass=("body_mass_g", "mean"),
    )
    .sort_values("mean_mass", ascending=False)
)

And a plot.

**In R**

```r
library(ggplot2)

ggplot(penguins, aes(flipper_length_mm, body_mass_g, color = species)) +
  geom_point()
```

**In Python**

`seaborn` is the closest standard Python tool to ggplot2. You hand it a data frame and say which columns map to x, y and colour (`hue` is seaborn's word for `color`).

In [ ]:
import seaborn as sns

ax = sns.scatterplot(data=penguins, x="flipper_length_mm", y="body_mass_g", hue="species")

seaborn draws its plots with `matplotlib`, the library underneath nearly all Python plotting. Chapter 05 covers both.

## Three things that will catch you out first

1. **Counting starts at 0.** `x[0]` is the first element, and `x[1:3]` gives the second and third.
2. **Indentation is syntax.** Python uses indentation where R uses `{ }`. A stray space at the start of a line is an error.
3. **`.` is not part of a name.** `my.data` is a fine variable name in R; in Python it means "the `data` attribute of `my`". Use `my_data`.

Chapter 01 covers all three, along with the rest of the core language.

---
Next: `01_language_basics.ipynb`